Instalación para Transformers

In [ ]:
# Esto es pesado (~2GB la primera vez que descarga el modelo)
pip install transformers torch datasets accelerate

# Si tienes GPU NVIDIA:
# pip install torch --index-url https://download.pytorch.org/whl/cu118

# Para verificar si tienes GPU disponible:
python -c "import torch; print(f'CUDA disponible: {torch.cuda.is_available()}')"

 Cargar datos para el Transformer

 Los Transformers NO usan TF-IDF. Tienen su propio tokenizador
que convierte texto en IDs numéricos que el modelo entiende.

"profit fell sharply" → [2082, 3062, 13559] (IDs del vocabulario)

La diferencia clave: el tokenizador mantiene el ORDEN y la
POSICIÓN de cada palabra. TF-IDF perdía eso.

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# 1. Cargar datos (mismo CSV de Kaggle)
df = pd.read_csv("all-data.csv",
                 encoding="latin-1",
                 header=None,
                 names=["sentiment", "sentence"])

# 2. Convertir etiquetas a números
#    Los Transformers necesitan labels numéricos
label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {v: k for k, v in label2id.items()}
df["label"] = df["sentiment"].map(label2id)

# 3. Train / Validation / Test split
#    Ahora hacemos 3 partes: train para entrenar, val para ajustar
#    hiperparámetros DURANTE el entrenamiento, test para evaluar al final.
#    ¿Por qué val? Porque el Transformer entrena varias "épocas" y
#    necesitamos saber cuándo parar (si no, sobreajusta).
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

print(f"Train: {len(train_df)} frases")
print(f"Val:   {len(val_df)} frases")
print(f"Test:  {len(test_df)} frases")

# 4. Convertir a formato HuggingFace Dataset
#    Es como un DataFrame pero optimizado para entrenar modelos
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[["sentence", "label"]].reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df[["sentence", "label"]].reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df[["sentence", "label"]].reset_index(drop=True)),
})

print(f"\nDataset creado:")
print(dataset)

# Veamos un ejemplo
print(f"\nEjemplo del dataset:")
print(dataset["train"][0])

BLOQUE 11 — Tokenización
Este bloque es donde el Transformer empieza a diferenciarse. Su tokenizador no solo parte en palabras — usa subword tokenization: puede partir una palabra desconocida en trozos que sí conoce.
python"""
BLOQUE 11: Tokenización con el tokenizador de FinBERT
======================================================
FinBERT es un BERT pre-entrenado en textos financieros.
Ya "sabe" lenguaje financiero antes de que le enseñemos nada.

Su tokenizador usa WordPiece:
  "restructuring" → ["restructuring"]  (la conoce entera)
  "outperformance" → ["out", "##perform", "##ance"]  (la parte)

Esto significa que NUNCA hay palabras "desconocidas", a diferencia
de TF-IDF donde si una palabra no estaba en train, se ignora.
"""
from transformers import AutoTokenizer

# Descargar tokenizador de FinBERT (~500MB primera vez)
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Veamos cómo tokeniza
examples = [
    "Profit increased significantly in the third quarter",
    "Operating losses widened due to restructuring charges",
    "The annual general meeting will be held next month",
]

print("CÓMO TOKENIZA FINBERT:")
print("=" * 60)
for text in examples:
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.encode(text)
    print(f"\n  Texto:   {text}")
    print(f"  Tokens:  {tokens}")
    print(f"  IDs:     {ids}")
    print(f"  N tokens: {len(tokens)}")

# Tokenizar todo el dataset
# padding=True → todas las frases tendrán la misma longitud (rellena con ceros)
# truncation=True → corta frases demasiado largas
# max_length=128 → máximo 128 tokens (suficiente para titulares)
def tokenize_function(examples):
    return tokenizer(
        examples["sentence"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Aplicar a todo el dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

print(f"\nDataset tokenizado:")
print(f"  Columnas: {tokenized_dataset['train'].column_names}")
print(f"  Ejemplo de input_ids (primeros 20): {tokenized_dataset['train'][0]['input_ids'][:20]}")
print(f"  Longitud de input_ids: {len(tokenized_dataset['train'][0]['input_ids'])}")